# Magpie Client Pipeline Runner — Colab (manual)

Manual, one-client-at-a-time runner for the same logic as the [Streamlit app](https://github.com/diegoakila/client_runner). No scheduling: you run each cell yourself, in order, whenever you want to (re)generate a client's data for a given month.

**Flow:** setup & auth → list clients → pick client + month → (optional) edit query → dry-run → confirm & run.

Running for real deletes any existing rows for that month in the destination table, then appends the fresh result — no duplicates within a month, and every other month is left untouched. Nothing is written to BigQuery until you explicitly set `CONFIRM = True` in the last cell.

## 1. Setup & auth

This clones the repo (so we can reuse `pipeline_core.py` instead of duplicating the logic here) and authenticates using your own Google identity via Colab's built-in auth flow — no credentials are stored in this notebook.

In [ ]:
import os

if not os.path.isdir("client_runner"):
    !git clone https://github.com/diegoakila/client_runner.git
%cd client_runner
!git pull -q
!pip install -q google-cloud-bigquery pandas db-dtypes

from google.colab import auth
auth.authenticate_user()

import pipeline_core as core

bq = core.get_client()
print("Connected to project:", core.PROJECT_ID)

## 2. List available clients

Loads the latest generated row per client from `pipeline.client_query`.

In [ ]:
clients_df = core.load_clients(bq)
clients_df[["client", "country", "dataset", "dest_table", "generated_at"]]

## 3. Pick a client and a month

Type a `client` value from the table above and the month you want to append (always the 1st of the month, `YYYY-MM-01`).

In [ ]:
client_name = "leekumkee"  #@param {type:"string"}
month_str = "2026-06-01"  #@param {type:"string"}

match = clients_df[clients_df["client"] == client_name]
assert not match.empty, f"Unknown client '{client_name}'. See the list printed in step 2."
row = match.iloc[0]

dataset = row["dataset"]
dest_table = row["dest_table"]
country = row["country"]
base_query = row["query"]

print("Dataset:    ", dataset)
print("Dest table: ", dest_table)
print("Generated:  ", row["generated_at"])

## 4. (Optional) edit the query

Leave `edited_query` as-is to run the stored version unchanged. Edit the string below only if you need a one-off change — if you do, and later confirm the run in step 6, the edited version is saved as a new row in `pipeline.client_query` so the history stays intact.

In [ ]:
edited_query = base_query  # edit this string if needed

scoped_query = core.build_scoped_query(edited_query, month_str)
print(scoped_query[:1500], "\n..." if len(scoped_query) > 1500 else "")

## 5. Dry-run (always do this before running for real)

Shows the estimated bytes processed, how many rows would be appended, whether rows for this month already exist in the destination table (they'll be deleted and replaced, not duplicated), and a 50-row sample.

In [ ]:
bytes_processed = core.dry_run(bq, scoped_query)
n_rows = core.row_count(bq, scoped_query)
existing_n = core.existing_month_count(bq, dest_table, month_str)
sample_df = core.preview_rows(bq, scoped_query, limit=50)

print(f"Estimated data processed: {bytes_processed / 1024**2:,.1f} MB")
print(f"Rows that would be appended: {n_rows:,}")
if existing_n > 0:
    print(f"{existing_n:,} rows already exist for {month_str} in {dest_table}. "
          "Running will delete those first, then append the fresh result (no duplicates).")
elif existing_n == 0:
    print(f"No existing rows for {month_str} in {dest_table} yet.")
else:
    print("Could not check existing rows (destination table may not exist yet).")

sample_df

## 6. Run for real (replace this month in `_dev`)

This only runs if `CONFIRM` is set to `True`. Set it, then re-run this cell. This deletes any existing rows for `month_str` in the client's `_dev` table, then appends the fresh result — no duplicates within the month, and every other month is left untouched.

In [ ]:
CONFIRM = False  #@param {type:"boolean"}

if not CONFIRM:
    print("Nothing was run. Set CONFIRM = True above and re-run this cell to actually run.")
else:
    if edited_query.strip() != base_query.strip():
        core.save_edited_query(bq, client_name, country, dataset, dest_table, edited_query)
        print("Edited query saved as a new row in pipeline.client_query")

    rows_deleted, job = core.replace_month(bq, scoped_query, dest_table, month_str)
    affected = job.num_dml_affected_rows if job.num_dml_affected_rows is not None else n_rows
    if rows_deleted > 0:
        print(f"Deleted {rows_deleted:,} existing rows for {month_str} in {dest_table}.")
    print(f"Appended {affected:,} rows into {dest_table} for month {month_str}.")

## 7. (Optional) Run for ALL clients at once — batch mode

Separate from the single-client flow above (steps 3–6). This loops over every client currently in `pipeline.client_query`, for one chosen month, dry-runs each one, and shows a summary table. It only runs after you explicitly confirm — same as the single-client flow. For each client, running deletes any existing rows for that month before appending the fresh result (no duplicates).

**Note:** this skips the per-client 50-row sample review from step 5 — you only see row counts and existing-row info per client, not the actual data. It also always uses each client's stored query as-is (no editing). If you want to eyeball a specific client's data, or run an edited query, use the single-client flow above instead.

In [ ]:
import pandas as pd

batch_month_str = "2026-06-01"  #@param {type:"string"}

all_clients_df = core.load_clients(bq)

batch_results = []
for _, r in all_clients_df.iterrows():
    c_name = r["client"]
    c_dest = r["dest_table"]
    c_query = r["query"]
    try:
        c_scoped = core.build_scoped_query(c_query, batch_month_str)
        c_bytes = core.dry_run(bq, c_scoped)
        c_rows = core.row_count(bq, c_scoped)
        c_existing = core.existing_month_count(bq, c_dest, batch_month_str)
        batch_results.append({
            "client": c_name,
            "dest_table": c_dest,
            "mb_processed": round(c_bytes / 1024**2, 1),
            "rows_to_append": c_rows,
            "existing_rows_for_month": c_existing,
            "status": "ok",
        })
    except Exception as e:
        batch_results.append({
            "client": c_name,
            "dest_table": c_dest,
            "mb_processed": None,
            "rows_to_append": None,
            "existing_rows_for_month": None,
            "status": f"dry-run failed: {e}",
        })

batch_summary_df = pd.DataFrame(batch_results)
print(f"Dry-ran {len(batch_summary_df)} clients for month {batch_month_str}.")
print("Rows marked 'existing_rows_for_month' > 0 will be deleted and replaced (not duplicated) when you run.")
batch_summary_df

In [ ]:
BATCH_CONFIRM = False  #@param {type:"boolean"}

if not BATCH_CONFIRM:
    print(
        f"Nothing was run. Review the summary table above for month {batch_month_str}, "
        "set BATCH_CONFIRM = True, and re-run this cell to run ALL clients."
    )
else:
    for _, r in all_clients_df.iterrows():
        c_name = r["client"]
        c_dest = r["dest_table"]
        c_query = r["query"]
        c_scoped = core.build_scoped_query(c_query, batch_month_str)
        try:
            rows_deleted, job = core.replace_month(bq, c_scoped, c_dest, batch_month_str)
            affected = job.num_dml_affected_rows
            deleted_note = f", deleted {rows_deleted:,} existing" if rows_deleted > 0 else ""
            print(f"[OK]     {c_name}: appended {affected if affected is not None else '?'} rows into {c_dest}{deleted_note}")
        except Exception as e:
            print(f"[FAILED] {c_name}: {e}")